# Model1: training + Kaggle submission
**Run:** set `DATA` to the folder with the raw data (files are searched recursively by name) and run all cells. Needs `Historical_*`, `Synthetic_water_value_*`, `Constraint_*`, `Unit_commitment_decisions.csv`, `Tokke_Vinje_topology.yaml`, `prediction_mapping.csv`, `sample_submission.csv`.

In [1]:
import numpy as np, pandas as pd, xgboost as xgb, yaml
from pathlib import Path
from numpy.lib.stride_tricks import sliding_window_view as swv

DATA = Path('./Users/lucaag/data/')                      # root folder of the raw data (searched recursively)
OUT = Path('submission_v11.csv')
T, ETA = 168, 0.88                    # horizon (hours), hydraulic efficiency
NE, FOLDS, GAP, SEEDS = 200, 5, 7, 3  # trees, CV folds for the OOF stacking features, fold gap (days), seeds averaged
# variant per plant, selected by OOF AUC in the v11 comparison: C = base, S = + surrogate, K = + constraints
VAR = {'vesle_kjela': 'S', 'songa': 'K', 'byrte': 'S', 'haukeli': 'K', 'kjela': 'S',
       'lio': 'K', 'vinje': 'K', 'tokke': 'K', 'hogga': 'K'}

def find(name):
    hits = sorted(DATA.rglob(name)); assert hits, f'{name} not found under {DATA.resolve()}'
    return hits[0]

## Data

In [2]:
snake = lambda c: c.lower().replace(' ', '_')

def read(name, idx='date'):
    d = pd.read_csv(find(name)); d[idx] = pd.to_datetime(d[idx], utc=True)
    return d.set_index(idx)

price = read('Historical_day_ahead_price_2015_2025.csv').set_axis(['day_ahead_price'], axis=1)
daily = lambda f: read(f).rename(columns=snake).reindex(price.index, method='ffill')  # daily series -> hourly
X = (price.join(read('Historical_inflow_1958_2025.csv').rename(columns=snake).add_suffix('_inflow'))
          .join(daily('Historical_volume_2015_2024.csv').add_suffix('_volume'))
          .join(daily('Synthetic_water_value_2015_2024.csv').add_suffix('_wv'))).sort_index()
Y = read('Unit_commitment_decisions.csv', 'starttime').rename(columns=snake).sort_index()
Y = Y.filter(like='result_committed').astype(np.uint8)   # drops Run No / total_calculation_time
C = pd.concat([read('Constraint_min_flow.csv'), read('Constraint_min_volume.csv')], axis=1).astype(np.float32)
topo = yaml.safe_load(find('Tokke_Vinje_topology.yaml').read_text(encoding='utf-8'))['model']

def cases(X, index):
    """One row per case (day): 168h window of price/inflows + initial volumes + water values at start and end of the week."""
    hourly = ['day_ahead_price'] + [c for c in X.columns if c.endswith('_inflow')]
    vol, wv = [[c for c in X.columns if c.endswith(s)] for s in ('_volume', '_wv')]
    pos = X.index.get_indexer(index); assert (pos >= 0).all() and (pos + T < len(X)).all()
    xh = swv(X[hourly].to_numpy(np.float32), T, axis=0)[pos].reshape(len(pos), -1)
    w = X[wv].to_numpy()
    xs = np.hstack([X[vol].to_numpy()[pos], w[pos], w[pos + T]])
    cols = [f'{c}_t{h}' for c in hourly for h in range(T)] + vol + [f'{c}_init' for c in wv] + [f'{c}_final' for c in wv]
    return pd.DataFrame(np.hstack([xh, xs]).astype(np.float32), index=index, columns=cols)

mp = pd.read_csv(find('prediction_mapping.csv'))
x_all = cases(X, Y.index)
x_kag = cases(X, pd.DatetimeIndex(pd.to_datetime(mp['starttime'], utc=True)))
print(x_all.shape, x_kag.shape, Y.shape)

(2922, 3243) (725, 3243) (2922, 2352)


## Topology and targets

In [3]:
RES = {k.lower(): v for k, v in topo['reservoir'].items()}
PL = {k.lower().replace(' ', '_'): v for k, v in topo['plant'].items()}
PN = {k.lower().replace(' ', '_'): g['p_nom'] for k, g in topo['generator'].items()}
# plant -> (upstream reservoirs, downstream reservoir)
TOPO = {'vesle_kjela': (['kjelavatn'], 'foersvatn'), 'kjela': (['foersvatn', 'bordalsvatn'], 'hyljelihyl'),
        'songa': (['songavatn', 'bitdalsvatn'], 'totak'), 'vinje': (['vaamarvatn', 'totak'], 'vinjevatn'),
        'haukeli': (['vatjern', 'langeidvatn'], 'vinjevatn'), 'tokke': (['vinjevatn'], 'bandak'),
        'byrte': (['botnedalsvatn'], 'byrtevatn'), 'lio': (['byrtevatn'], 'bandak'), 'hogga': (['bandak'], None)}
UP_PLANTS = {'kjela': ['vesle_kjela'], 'vinje': ['kjela', 'songa'], 'tokke': ['vinje', 'haukeli'],
             'hogga': ['tokke', 'lio'], 'lio': ['byrte']}
SRC = {'vinje': 'totak'}
ORDER = ['vesle_kjela', 'songa', 'byrte', 'haukeli', 'kjela', 'lio', 'vinje', 'tokke', 'hogga']  # upstream first

units = sorted({c[len('result_committed_'):].rsplit('_t', 1)[0] for c in Y.columns})
plant_units = {p: [i for i, u in enumerate(units) if u.rsplit('_g', 1)[0] == p] for p in ORDER}
ucols = lambda u: [f'result_committed_{units[u]}_t{h}' for h in range(T)]
Y3 = np.stack([Y[ucols(u)].to_numpy() for u in range(len(units))], axis=1)   # (cases, units, 168)
Yflat = [Y3[:, u, :].ravel() for u in range(len(units))]
out_cols = [c for u in range(len(units)) for c in ucols(u)]

## Features (base, surrogate dispatch, constraints)

In [4]:
PRICE = [f'day_ahead_price_t{h}' for h in range(T)]

def shift(a, k):  # price at t-k (edge-padded)
    o = np.roll(a, k, 1)
    if k > 0: o[:, :k] = a[:, :1]
    else: o[:, k:] = a[:, -1:]
    return o

def head(r, v):  # water level of a reservoir from its volume
    d = RES[r]['vol_head']; return np.interp(v, d['x'], d['y'])

def static(X, plant):  # per-case features: volumes, water values, filling, head
    up, dn = TOPO[plant]; rs = up + ([dn] if dn else [])
    S = [X[[f'{r}_{s}' for r in rs for s in ('volume', 'wv_init', 'wv_final')]].to_numpy()]
    S += [(X[f'{r}_volume'] / RES[r]['max_vol']).to_numpy()[:, None] for r in rs]
    hu = head(up[0], X[f'{up[0]}_volume'].to_numpy())
    hd = head(dn, X[f'{dn}_volume'].to_numpy()) if dn else np.zeros_like(hu)
    wd = X[f'{dn}_wv_final'].to_numpy() if dn else np.zeros(len(X))
    S.append(np.column_stack([hu, hu - np.maximum(hd, PL[plant]['outlet_line']), X[f'{up[0]}_wv_final'].to_numpy() - wd]))
    return np.hstack(S)

def inflow(X, up):  # hourly inflow summed over the upstream reservoirs (m3/s)
    return sum((X[[f'{r}_inflow_t{h}' for h in range(T)]].to_numpy() for r in up if f'{r}_inflow_t0' in X.columns),
               np.zeros((len(X), T), np.float32))

def cap(X, plant):  # max flow of the plant from nominal power and head
    up, dn = TOPO[plant]
    hu = head(up[0], X[f'{up[0]}_volume'].to_numpy())
    hd = head(dn, X[f'{dn}_volume'].to_numpy()) if dn else 0
    H = np.maximum(hu - np.maximum(hd, PL[plant]['outlet_line']), 5)
    return sum(PN[units[u]] for u in plant_units[plant]) / (2.725 * ETA * H)

CC, _cw = list(C.columns), {}
def cwin(X):  # 168h windows of the min-flow / min-volume constraints for each case
    k = (X.index[0], len(X))
    if k not in _cw:
        pos = C.index.get_indexer(X.index); assert (pos >= 0).all()
        _cw[k] = swv(C.to_numpy(), T, axis=0)[pos]
    return _cw[k]
mvol = lambda X, rs: sum((cwin(X)[:, CC.index(r.capitalize())].max(1) for r in rs if r.capitalize() in CC), np.zeros(len(X), np.float32))
CONS = {'lio': ['r_Vest_Vassdraget', 'b_Byrtevatn', 'b_Vinjevatn', 'Byrtevatn'], 'byrte': ['b_Byrtevatn', 'Byrtevatn'], 'songa': ['Totak'],
        'vinje': ['Totak', 'b_Vinjevatn'], 'vesle_kjela': ['b_Kjelavatn'], 'kjela': ['b_Hyljelihyl'], 'tokke': ['b_Vinjevatn'], 'hogga': ['b_Bandak']}

def surr(X, plant, rel=None, cons=False):
    """Greedy water-filling: pour a weekly water budget (min = no spill, max = available) into the highest-price hours."""
    up, c, nu = TOPO[plant][0], cap(X, plant), len(plant_units[plant])
    src = {**SRC, **({'haukeli': 'langeidvatn'} if cons else {})}.get(plant, up[0])
    V0 = sum(X[f'{r}_volume'].to_numpy() for r in up)
    Win = inflow(X, up).sum(1) * 0.0036 + (rel.sum(1) if rel is not None else 0)  # m3/s -> Mm3/h, + release of upstream plants
    Wmax = np.clip(V0 + Win - (mvol(X, up) if cons else 0), 0, T * c)
    Wmin = np.minimum(np.maximum(V0 + Win - sum(RES[r]['max_vol'] for r in up), 0), Wmax)
    P = X[PRICE].to_numpy(); rank = (-P).argsort(1).argsort(1)
    Wp = [np.clip((P > X[f'{src}_{w}'].to_numpy()[:, None]).sum(1) * c, Wmin, Wmax) for w in ('wv_init', 'wv_final')]
    Ws = [Wmin, *Wp, Wmax]
    S = [np.clip((W / c)[:, None] - rank, 0, 1) * nu for W in Ws]
    S += [np.repeat((W / c / T)[:, None], T, 1) for W in Ws]
    if rel is not None: S += [rel / c[:, None] * nu, np.cumsum(rel, 1) / c[:, None]]
    return S

def cons_feats(X, plant, c):
    up, dn = TOPO[plant]; n, W = len(X), cwin(X)
    bc = lambda g: np.repeat(np.asarray(g, np.float32)[:, None], T, 1)
    Vd = X[f'{dn}_volume'].to_numpy() if dn else np.zeros(n)
    Id = inflow(X, [dn]).sum(1) * 0.0036 if dn else np.zeros(n)
    F = [W[:, CC.index(k)] for k in CONS.get(plant, [])]
    F += [bc(sum(X[f'{r}_volume'].to_numpy() for r in up) - mvol(X, up)),
          bc(np.maximum(mvol(X, [dn] if dn else []) - Vd - Id, 0) / c)]
    if plant == 'lio':
        w = lambda k: W[:, CC.index(k)]
        need = np.maximum(w('r_Vest_Vassdraget') - inflow(X, ['r_vest_vassdraget']) - w('b_Vinjevatn') - w('b_Byrtevatn'), 0)
        q = (c / 0.0036)[:, None]
        F += [need, np.cumsum(need, 1) / q, bc(need.sum(1) / q[:, 0])]
    return F

def base(X, plant, up_prob=None, rel=None, var='C'):
    """Long feature matrix (cases*168 rows) for one plant."""
    up, dn, n = TOPO[plant][0], TOPO[plant][1], len(X)
    P = X[PRICE].to_numpy(); D = P.reshape(n, 7, 24)
    I = inflow(X, up); wvf = X[f'{up[0]}_wv_final'].to_numpy()[:, None]
    h = np.tile(np.arange(T), (n, 1))
    F = [h, h % 24, (X.index.dayofweek.to_numpy()[:, None] + h // 24) % 7,
         P, shift(P, 1), shift(P, -1), P.argsort(1).argsort(1) / (T - 1),
         D.argsort(2).argsort(2).reshape(n, T) / 23, P - np.repeat(D.mean(2), 24, 1),
         P - P.mean(1, keepdims=True), P - wvf, I, np.repeat(I.mean(1, keepdims=True), T, 1)]
    if up_prob is not None:  # stacked: predicted on-probability of the upstream plants
        F += [up_prob, shift(up_prob, 1), shift(up_prob, 3), np.repeat(up_prob.mean(1, keepdims=True), T, 1)]
    if var == 'K':
        F += cons_feats(X, plant, cap(X, plant)) + [P - X[f'{r}_wv_{w}'].to_numpy()[:, None] for r in up + ([dn] if dn else []) for w in ('init', 'final')]
    A = np.column_stack([f.ravel() for f in F] + [np.repeat(static(X, plant), T, 0), np.repeat(X.index.dayofyear.to_numpy(), T)])
    if var != 'C':
        S = surr(X, plant, rel, var == 'K')
        A = np.column_stack([A] + [s.ravel() for s in S] + [S[2].ravel()])
    return A.astype(np.float32)

## Training
One XGBoost per plant (units stacked with a unit-index feature, `SEEDS` seeds averaged). The contiguous-fold CV (with gap) only produces the out-of-fold probabilities that feed the downstream plants; the final predictions come from the model fitted on all cases.

In [5]:
mk = lambda seed: xgb.XGBClassifier(n_estimators=NE, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                                    tree_method='hist', n_jobs=-1, random_state=seed)

def fp(X, y, Xs):
    if y.min() == y.max(): return [np.full(len(x), float(np.clip(y[0], .01, .99))) for x in Xs]
    ms = [mk(s).fit(X, y) for s in range(SEEDS)]
    return [np.mean([m.predict_proba(x)[:, 1] for m in ms], 0) for x in Xs]

def folds(n):
    b = np.linspace(0, n, FOLDS + 1, dtype=int)
    for a, c in zip(b[:-1], b[1:]): yield np.r_[0:max(a - GAP, 0), min(c + GAP, n):n], np.arange(a, c)

ridx = lambda c: (np.asarray(c)[:, None] * T + np.arange(T)).ravel()
ylong = lambda c, us: np.concatenate([Yflat[u][ridx(c)] for u in us])

def design(B, c, us, sur):  # stack the units of a plant vertically, adding unit id / index (and hour-minus-index for surrogates)
    r = ridx(c); R = B[r]
    blk = lambda j, u: np.column_stack([R, np.full(len(r), u), np.full(len(r), j)] + ([R[:, -1] - j] if sur else []))
    return np.vstack([blk(j, u) for j, u in enumerate(us)])

def fit_group(B, Bt, us, sur):
    n, nt = len(B) // T, len(Bt) // T
    allc = np.arange(n)
    te = fp(design(B, allc, us, sur), ylong(allc, us), [design(Bt, np.arange(nt), us, sur)])[0].reshape(len(us), nt, T)
    oo = np.zeros((len(us), n, T))
    for tr, va in folds(n):
        p = fp(design(B, tr, us, sur), ylong(tr, us), [design(B, va, us, sur)])[0]
        oo[:, va] = p.reshape(len(us), len(va), T)
    return oo, te

def run(VAR, xtr, xte):
    oof, te, rtr, rte = {}, {}, {}, {}
    P = np.zeros((len(xte), len(units), T), np.float32)
    for plant in ORDER:
        us, ups, v = plant_units[plant], UP_PLANTS.get(plant, []), VAR[plant]
        m = lambda d: np.mean([d[p] for p in ups], 0) if ups else None
        s = lambda d: sum(d[p] for p in ups) if (ups and v != 'C') else None
        oo, t = fit_group(base(xtr, plant, m(oof), s(rtr), v), base(xte, plant, m(te), s(rte), v), us, v != 'C')
        P[:, us, :] = t.transpose(1, 0, 2)
        te[plant], oof[plant] = t.mean(0), oo.mean(0)
        rtr[plant], rte[plant] = oof[plant] * cap(xtr, plant)[:, None], te[plant] * cap(xte, plant)[:, None]
        print(plant, 'ok')
    return P

P = run(VAR, x_all, x_kag)

vesle_kjela ok
songa ok
byrte ok
haukeli ok
kjela ok
lio ok
vinje ok
tokke ok
hogga ok


## Submission

In [6]:
sub = pd.DataFrame(P.reshape(len(P), -1), columns=out_cols)[list(Y.columns)]
sub.insert(0, 'Run No', mp['Run No'].to_numpy())
ss = pd.read_csv(find('sample_submission.csv'))
norm = {snake(c): c for c in ss.columns}
sub = sub.rename(columns=lambda c: norm.get(snake(c), c))          # column names as in sample_submission.csv
assert set(sub.columns) == set(ss.columns), set(sub.columns) ^ set(ss.columns)
sub = sub.set_index('Run No').loc[ss['Run No']].reset_index()[list(ss.columns)]
sub.to_csv(OUT, index=False)
print(sub.shape, sub.iloc[:, 1:].to_numpy().mean().round(3))

(725, 2353) 0.461
